# 09 — Conversation and Long-Context Engineering

## Scenario
A customer starts a chat, mentions their order number, gets distracted by asking a bunch of unrelated questions about shipping policies, and then finally asks for an update on their order.

**The Problem:** We cannot pass the *entire* history of every user into every prompt. It is too expensive, too slow, and leads to "Lost in the Middle" hallucinations. We need a strategy to persist state.

## Strategy 1: The Sliding Window

The simplest approach is to only pass the last N messages to the LLM to save tokens.

## Strategy 2: Summary Memory

Instead of truncating, we can ask an LLM to periodically summarize the conversation and pass the summary along with the sliding window.

## Strategy 3: Structured State Extraction (State of the Art)

The gold standard for production AI (used by agent frameworks) is to maintain a strict, structured "State Object". After every user message, an LLM extracts critical facts into this schema.

In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab09 import *

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Strategy 1: The Sliding Window

In [ ]:
case = CASES[0]
window = sliding_window(case["history"], budget_tokens=15)
print("WINDOW:", window)
assert contains_order_id("\n".join(window)) is None
request = next(r for r in build_requests() if r.case_id == "i09/window/refund-conversation")
print("PROMPT:", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)

## Strategy 2: Summary Memory

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i09/summary/refund-conversation")
print("PROMPT:", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
assert contains_order_id(response.text) == "ORD-5592"

## Strategy 3: Structured State Extraction

In [ ]:
extract = client.generate(next(r for r in build_requests() if r.case_id == "i09/state/extract-turn-1"))
state = UserState.model_validate_json(extract.text)
print("RECORDED STATE:", state)
assert state.active_order_id == "ORD-5592"
conflict = merge_state(state, UserState(active_order_id="ORD-7710"))
assert conflict.needs_confirmation

## Takeaway

This replay-backed experiment makes the application control and measured trade-off explicit.

## References

See the course README for the references and further reading.